# M3v2 Floor/Window — 정제 + 재학습 (A100 권장)

**Drive 준비:**
1. `floor_window.zip` (~1GB)
2. `m3_baseline_best.pt` (52MB, 기존 mAP 0.804)

**총 시간:** A100 batch=8 약 3~4시간

**클래스:** floor_defect, glass_defect, frame_defect (3)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')  # 본인 계정 폴더
ZIP_NAME = 'floor_window.zip'
BEST_PT_PATH = DRIVE_DIR / 'm3_baseline_best.pt'

WORK = Path('/content/m3v2')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
assert zip_path.exists(), f'{zip_path} 없음'
ds_dir = WORK / 'floor_window'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

assert BEST_PT_PATH.exists(), f'{BEST_PT_PATH} 없음'
best_pt = WORK / 'baseline_best.pt'
shutil.copy2(BEST_PT_PATH, best_pt)
print(f'baseline best.pt: {best_pt} ({best_pt.stat().st_size/1024/1024:.1f}MB)')
for s in ['train', 'val', 'test']:
    p = ds_dir / 'images' / s
    if p.exists():
        print(f'  {s}: {len(list(p.glob("*")))} images')

In [ ]:
import numpy as np
from ultralytics import YOLO

IOU_BAD = 0.3
CONF_HIGH = 0.85
MAX_MISSED = 5

def yolo_to_xyxy(b, w, h):
    cx, cy, bw, bh = b[1]*w, b[2]*h, b[3]*w, b[4]*h
    return [cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter == 0:
        return 0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

def read_lbl(p):
    if not p.exists():
        return []
    out = []
    for line in p.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            try:
                out.append([int(parts[0])] + [float(x) for x in parts[1:5]])
            except ValueError:
                pass
    return out

def write_lbl(p, boxes):
    if not boxes:
        if p.exists():
            p.unlink()
        return
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text('\n'.join(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}' for b in boxes))

refined = WORK / 'm3_refined'
if refined.exists():
    shutil.rmtree(refined)
model = YOLO(str(best_pt))
stats = {'total': 0, 'bad': 0, 'missed': 0}

for split in ['train', 'val', 'test']:
    src_img = ds_dir / 'images' / split
    src_lbl = ds_dir / 'labels' / split
    if not src_img.exists():
        continue
    dst_img = refined / 'images' / split
    dst_lbl = refined / 'labels' / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    images = [f for f in src_img.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    print(f'[{split}] {len(images)}')
    if split in ['val', 'test']:
        for im in images:
            shutil.copy2(im, dst_img / im.name)
            l = src_lbl / (im.stem + '.txt')
            if l.exists():
                shutil.copy2(l, dst_lbl / l.name)
        continue
    for i in range(0, len(images), 16):
        batch = images[i:i+16]
        results = model.predict(source=[str(b) for b in batch], conf=0.25, iou=0.5, imgsz=960, verbose=False, save=False)
        for img_path, res in zip(batch, results):
            stats['total'] += 1
            gt = read_lbl(src_lbl / (img_path.stem + '.txt'))
            pxy = res.boxes.xyxy.cpu().numpy() if len(res.boxes) > 0 else np.empty((0, 4))
            pcf = res.boxes.conf.cpu().numpy() if len(res.boxes) > 0 else np.empty(0)
            pcl = res.boxes.cls.cpu().numpy().astype(int) if len(res.boxes) > 0 else np.empty(0, dtype=int)
            W, H = res.orig_shape[1], res.orig_shape[0]
            cleaned = []
            for g in gt:
                gxy = yolo_to_xyxy(g, W, H)
                same = pcl == g[0]
                if same.sum() == 0:
                    cleaned.append(g)
                    continue
                ious = np.array([iou(gxy, pxy[j]) for j in np.where(same)[0]])
                if ious.max() >= IOU_BAD:
                    cleaned.append(g)
                else:
                    stats['bad'] += 1
            missed = 0
            for j in range(len(pxy)):
                if pcf[j] < CONF_HIGH:
                    continue
                m_ = False
                for g in cleaned:
                    if int(g[0]) != pcl[j]:
                        continue
                    if iou(yolo_to_xyxy(g, W, H), pxy[j]) >= IOU_BAD:
                        m_ = True
                        break
                if not m_ and missed < MAX_MISSED:
                    x1, y1, x2, y2 = pxy[j]
                    cleaned.append([int(pcl[j]), (x1+x2)/2/W, (y1+y2)/2/H, (x2-x1)/W, (y2-y1)/H])
                    stats['missed'] += 1
                    missed += 1
            shutil.copy2(img_path, dst_img / img_path.name)
            write_lbl(dst_lbl / (img_path.stem + '.txt'), cleaned)
        if (i+16) % 320 == 0 or i+16 >= len(images):
            print(f'  {min(i+16, len(images))}/{len(images)} (bad:{stats["bad"]}, missed:{stats["missed"]})')

yaml_text = f'''path: {refined}
train: images/train
val: images/val
test: images/test
nc: 3
names:
  0: floor_defect
  1: glass_defect
  2: frame_defect
'''
data_yaml = WORK / 'm3_refined.yaml'
data_yaml.write_text(yaml_text)
print(f'정제: bad={stats["bad"]}, missed={stats["missed"]}')

In [ ]:
train_imgs = sorted([f for f in (refined / 'images' / 'train').iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
losses = []
for i in range(0, len(train_imgs), 8):
    batch = train_imgs[i:i+8]
    results = model.predict(source=[str(b) for b in batch], conf=0.05, iou=0.5, imgsz=960, verbose=False, save=False)
    for img_path, res in zip(batch, results):
        gt = read_lbl(refined / 'labels' / 'train' / (img_path.stem + '.txt'))
        if not gt:
            losses.append((img_path, 0.0))
            continue
        if len(res.boxes) == 0:
            losses.append((img_path, 1.0))
            continue
        pxy = res.boxes.xyxy.cpu().numpy()
        pcf = res.boxes.conf.cpu().numpy()
        pcl = res.boxes.cls.cpu().numpy().astype(int)
        W, H = res.orig_shape[1], res.orig_shape[0]
        confs = []
        for g in gt:
            gxy = yolo_to_xyxy(g, W, H)
            best = 0
            for j in range(len(pxy)):
                if pcl[j] == g[0] and iou(gxy, pxy[j]) >= 0.3:
                    best = max(best, pcf[j])
            confs.append(best)
        losses.append((img_path, 1.0 - sum(confs)/len(confs)))
    if (i+8) % 320 == 0:
        print(f'  {min(i+8, len(train_imgs))}/{len(train_imgs)}')

losses.sort(key=lambda x: x[1], reverse=True)
n_hard = max(1, len(losses)//10)
for img_path, _ in losses[:n_hard]:
    lbl = refined / 'labels' / 'train' / (img_path.stem + '.txt')
    for k in range(4):
        shutil.copy2(img_path, refined / 'images' / 'train' / f'{img_path.stem}_h{k}{img_path.suffix}')
        if lbl.exists():
            shutil.copy2(lbl, refined / 'labels' / 'train' / f'{img_path.stem}_h{k}.txt')
print(f'Hard samples: {n_hard} 추가')

In [ ]:
import threading, time
AUTOSAVE = DRIVE_DIR / 'm3v2_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m3v2')
PROJECT.mkdir(parents=True, exist_ok=True)
stop_flag = [False]

def autosave():
    while not stop_flag[0]:
        time.sleep(300)
        for s in ['stage1', 'stage2']:
            for src in [PROJECT/s/'weights/last.pt', PROJECT/s/'weights/best.pt']:
                if src.exists():
                    try:
                        shutil.copy2(src, AUTOSAVE / f'{s}_{src.name}')
                    except Exception:
                        pass
        print(f'[autosave] {time.strftime("%H:%M:%S")}')

threading.Thread(target=autosave, daemon=True).start()

print('=== Stage 1 (35ep, lr=1e-3, A100 batch=8) ===')
model_s1 = YOLO('yolo11m.pt')
model_s1.train(
    data=str(data_yaml),
    epochs=35,
    batch=8,
    imgsz=1280,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,
    patience=12,
    warmup_epochs=3,
    close_mosaic=15,
    freeze=0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5,
    shear=2.0, perspective=0.001,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.15, copy_paste=0.5,
    erasing=0.0,
    multi_scale=0.3,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage1',
    exist_ok=True
)

print('\n=== Stage 2 (10ep, lr=1e-5, freeze=10) ===')
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
model_s2 = YOLO(str(stage1_best))
model_s2.train(
    data=str(data_yaml),
    epochs=10,
    batch=8,
    imgsz=1280,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-5,
    lrf=0.01,
    cos_lr=True,
    patience=6,
    warmup_epochs=1,
    close_mosaic=8,
    freeze=10,
    mosaic=0.5, mixup=0.0, copy_paste=0.2,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage2',
    exist_ok=True
)
stop_flag[0] = True

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
stage2_best = PROJECT / 'stage2' / 'weights' / 'best.pt'
best_path = stage2_best if stage2_best.exists() else stage1_best
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')
metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=8)
print(f'\n=== M3v2 최종 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'0.9? {"YES" if metrics.box.map50 >= 0.9 else "NO"}')
OUT = DRIVE_DIR / 'm3v2_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm3_yolo_floor_window.onnx')
shutil.copy2(best_path, OUT / 'm3v2_best.pt')
for s in ['stage1', 'stage2']:
    rcsv = PROJECT / s / 'results.csv'
    if rcsv.exists():
        shutil.copy2(rcsv, OUT / f'{s}_results.csv')
print('Saved:', OUT)